# RAG - Question Answering on the 2025 Threat Assessment

Use a RAG pipeline to answer complex geopolitical questions using the 2025 Annual Threat Assessment of the U.S. Intelligence Community.

### 🔧 Install Dependencies
Install the required libraries below. If you're running in Colab, uncomment the lines and run the cell.

In [ ]:
# !pip install -U langchain langchain-community langchain-text-splitters
# !pip install -U chromadb sentence-transformers pymupdf
# !pip install -U transformers accelerate

In [ ]:
DEVICE = "cuda"   # "cpu"

### 🔍 Inspect the PDF Document

Browse through the document. Take note of those things we've discussed. Which pages would you drop?

We should probably ignore the first three pages and the last one.

### 📄 Load the PDF Document
We will use `PyMuPDFLoader` to load the uploaded PDF.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

In [ ]:
PDF_DOCUMENT_PATH = "assets/2025 Annual Threat Assessment of the U.S. Intelligence Community.pdf"

In [ ]:
loader = PyMuPDFLoader(PDF_DOCUMENT_PATH)
docs = loader.load()

In [ ]:
print(f"Loaded {len(docs)} pages")
print(docs[0].page_content[:500])

Remove unwanted pages:

In [ ]:
unwanted_pages = (0, 1, 2, 33)

In [ ]:
docs = [docs[page] for page in range(len(docs)) if page not in unwanted_pages]

In [ ]:
print(docs[0].page_content[:500])

### ✂️ Chunking
Split the loaded pages into manageable chunks using `SpacyTextSplitter` (by sentence).

We're using a `chunk_size` of **1,500 characters**, which approximates **270 words** – which fits `bge-small-en-v1.5` embedding model's input context nicely (512-tokens max), with room to spare.

We also use a small `chunk_overlap` to maintain continuity between adjacent chunks.

In [ ]:
from langchain_text_splitters import SpacyTextSplitter

In [ ]:
# Chunk size should fit within the embedding model's input context.
# all-MiniLM-L6-v2:  256 input tokens --> chunk_size=1000
# bge-small-en-v1.5: 512 input tokens --> chunk_size=1500
splitter = SpacyTextSplitter(chunk_size=1500, chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [ ]:
chunks[0]

### 🧠 Vector Store and Embedding
Create a ChromaDB collection and embed the text using sentence-transformers.

In [ ]:
import hashlib

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

In [ ]:
client = chromadb.Client()

# embedding_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device=DEVICE)      # 384 dims / 256 tokens
embedding_fn = SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-small-en-v1.5", device=DEVICE)  # 384 dims / 512 tokens

collection = client.get_or_create_collection(
    "threat-assessment", 
    embedding_function=embedding_fn,
    configuration={
        "hnsw": {
            "space": "cosine"
        },
    }
)

In [ ]:
for doc in chunks:
    content = doc.page_content
    doc_id = hashlib.md5(content.encode()).hexdigest()[-8:]
    collection.add(ids=[doc_id], documents=[content])

In [ ]:
collection.count()

### 🔍 Ask a Question
We will ask a question and retrieve relevant chunks.

In [ ]:
user_query = "How is China using AI and biotechnology to challenge U.S. dominance globally?"

In [ ]:
results = collection.query(query_texts=[user_query], n_results=5, include=["documents"])
context_chunks = results["documents"][0]

In [ ]:
print("\n---\n".join(context_chunks))

### 📝 Prompt Construction

Build the LLM prompt using the retrieved context.

In [ ]:
prompt_template = """You are a geopolitical analyst. Based on the context below, answer the user's question clearly and concisely.

Context:
{context}

Question:\n
{user_query}

Answer:"""

In [ ]:
context = "\n---\n".join(context_chunks)

In [ ]:
prompt = prompt_template.format(context=context, user_query=user_query)
print(prompt)

### 🤖 Text Generation
Use a Hugging Face LLM to generate the final answer.

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import torch

In [ ]:
model_id = "unsloth/Llama-3.2-1B-Instruct"
# model_id = "Qwen/Qwen3-0.6B"
# model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map="auto")

In [ ]:
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
generation_config = GenerationConfig(
    max_new_tokens=1000,
    temperature=0.0,
    do_sample=False,   # <--- if we insist on 0.0 temperature
    clean_up_tokenization_spaces=False,
)

In [ ]:
response = generator(
    text_inputs=prompt, 
    generation_config=generation_config,
    return_full_text=False,
)

In [ ]:
print(response[0]["generated_text"])

### 📝 Alternative Prompt Construction

We used plain chunk concatenation, now let's try a different strategy - Map-Reduce. The process builds on what we've already done:
1. Map phase: run the LLM (same prompt as above) on each retrieved chunk separately.
2. Reduce phase: aggregate all the answers into a final response (using the prompt below).

In [ ]:
reduce_prompt = """You are a geopolitical analyst. Partial answers to a user's question are given. 
Based on the partial answers, give a final complete, comprehensive, answer.

Question:\n
{user_query}

Partial answers:\n
{answers}

Final answer:"""

In [ ]:
partial_answers = []
for chunk in context_chunks:
    prompt = prompt_template.format(context=chunk, user_query=user_query)
    response = generator(prompt, max_new_tokens=300, return_full_text=False, temperature=0.01)
    partial_answers.append(response[0]["generated_text"])

In [ ]:
print(f"\n{'~' * 100}\n".join(partial_answers))

In [ ]:
final_prompt = reduce_prompt.format(user_query=user_query, answers="\n---\n".join(partial_answers))
print(final_prompt)

In [ ]:
response = generator(final_prompt, max_new_tokens=500, return_full_text=False, temperature=0.01)

In [ ]:
print(response[0]["generated_text"])

### ✅ Done!
Try changing the question or chunking parameters to explore other areas of the report.

<hr/>